In [ ]:
!pip install transformers langchain langchain_community torch gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is 

In [ ]:
pip install langchain langchain-community langchain-core


In [ ]:
pip install langchain-classic


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_classic.memory import ConversationBufferMemory
import gradio as gr

In [ ]:
# load model and tokenzier
model_name= "google/gemma-4-31B-it"
token=""

In [ ]:
#check
device=torch.device ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [ ]:
pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 99.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    model_name
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=token   # updated argument name
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
def chatbot_response(user_input):
  inputs=tokenizer(user_input, return_tensors="pt",max_length=512,truncation=True).to(device)

  outputs=model.generate(**inputs)

In [ ]:
def chatbot_response(user_input):
    # 1. Prepare the input
    inputs = tokenizer(user_input, return_tensors="pt", max_length=512, truncation=True).to(device)

    # 2. Generate the response
    outputs = model.generate(**inputs)

    # 3. Decode the response
    bot_reply = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 4. Clean up the reply (Gemma often repeats the prompt)
    if bot_reply.startswith(user_input):
        bot_reply = bot_reply[len(user_input):].strip()

    return bot_reply

In [ ]:
def chatbot_response(user_input,histroy):
  resposne=chatbot_response(user_input)
  history.append((user_input,response))
  return history,""

demo=gr.Blocks()


In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## Gen AI Chatbot")
    chatbot = gr.Chatbot()

    with gr.Row():
        user_input = gr.Textbox(placeholder="Type your message here...", lines=1)
        send_button = gr.Button("Send")

    # Corrected: Capital 'S' and parentheses for the list
    history = gr.State([])

    # Ensure the function name matches what you defined in the previous cell
    send_button.click(chatbot_response,
                      inputs=[user_input, history],
                      outputs=[chatbot, user_input])

    user_input.submit(chatbot_response,
                      inputs=[user_input, history],
                      outputs=[chatbot, user_input])

demo.launch(share=True)